# This notebook is for the skill predictor for Starcraft Skill

In [ ]:
# Imports
import numpy as np 
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [ ]:
# Load Dataset

# Read the raw CSV file into a list of lines.
with open('starcraft/train.csv') as f:
    lines = f.read().split('\n')
 
# Assign a compact numeric index to each unique player name.
p = 0
playerid = {}
for line in lines:
    row = line.split(',')
    if len(row) != 10:
        continue
    for name in (row[1], row[4]):
        if name not in playerid:
            playerid[name] = p
            p += 1
 
# Number of unique players and a reverse lookup for names.
nplayers = len(playerid)
playername = [''] * nplayers
for name, pid in playerid.items():
    playername[pid] = name
 
# Build win/play counts and store full game records for later use.
pKeep = 1.0   # fraction of game records to include (1.0 = keep all)
nEdge = 3     # target number of opponents per player
nKeep = 5     # maximum number of games to keep per player pair
 
nplays = np.zeros((nplayers, nplayers))
nwins  = np.zeros((nplayers, nplayers))
 
# Each entry: (id_a, id_b, a_won:float, class_a:str, class_b:str, venue:str)
game_records = []
all_classes  = set()
all_venues   = set()
 
for line in lines:
    row = line.split(',')
    if len(row) != 10:
        continue
 
    # Extract both players and result fields.
    a_name, b_name = row[1], row[4]
    a_won          = row[2] == '[winner]'
    b_won          = row[5] == '[winner]'
    is_draw        = a_won and b_won
    class_a        = row[6].strip()
    class_b        = row[7].strip()
    venue          = row[9].strip()
 
    # Track all seen unit races and venues
    all_classes.update([class_a, class_b])
    all_venues.add(venue)
 
    a, b = playerid[a_name], playerid[b_name]
 
    # Filter game records by random sampling, nkeep, and nedge criteria.
    # In case of ties, credit each player with a half-win and half-loss.
    if np.random.rand() < pKeep:
        if (nplays[a, b] < nKeep) and \
           (((nplays[a, :] > 0).sum() < nEdge) or ((nplays[:, b] > 0).sum() < nEdge)):
            nplays[a, b] += 1
            nplays[b, a] += 1
            nwins[a, b] += 0.5 if is_draw else float(a_won)
            nwins[b, a] += 0.5 if is_draw else float(b_won)
            game_records.append((a, b, 0.5 if is_draw else float(a_won), class_a, class_b, venue))
 
# Build lookup tables for discrete context features.
classes   = sorted(all_classes)
venues    = sorted(all_venues)
class_idx = {c: i for i, c in enumerate(classes)}
venue_idx = {v: i for i, v in enumerate(venues)}
NC, NV    = len(classes), len(venues)
 
print(f"Players : {nplayers}")
print(f"Classes : {classes}")
print(f"Venues  : {venues}")
print(f"Games   : {len(game_records)}")

# Installments
hots = heart of the swarm (Z)

lotv = legacy of the void (P) ( newest and which most people play on )

wol = wings of liberty (T) ( oldest )
# Classes
Protoss (P)

Terran (T)

Zerg (Z)

(07/17/2016,MC,[loser],0–1,Cure,[winner],P,T,LotV,offline)

(Example entry)

In [ ]:
# Model definition

def build_markov_transition(nplays: np.ndarray,
                             nwins:  np.ndarray,
                             damping: float = 0.15) -> np.ndarray:
    """
    Build a row-stochastic Markov transition matrix.
    T[i, j] = fraction of i's prestige that flows to j per step.
    Skill/prestige flows to whoever wins: if j beats i, T[i,j] is large.
    The damping term (uniform jump) guarantees ergodicity (unique stationary dist.).
    """
    n = nplays.shape[0]
    T = np.zeros((n, n))
    for i in range(n):
        for j in np.where(nplays[i, :] > 0)[0]:
            if i == j:
                continue
            T[i, j] = nwins[j, i] / nplays[i, j]   # j's win rate against i
        row_sum = T[i, :].sum()
        if row_sum > 0:
            T[i, :] /= row_sum
        else:
            # Isolated player: distribute uniformly to all players.
            T[i, :] = 1.0 / n
    return (1.0 - damping) * T + (damping / n) * np.ones((n, n))
 
 
def power_method(T: np.ndarray, max_iter: int = 2000, tol: float = 1e-10) -> np.ndarray:
    """Find stationary distribution of T by iterating π ← π T."""
    n = T.shape[0]
    skill = np.ones(n) / n
    for it in range(max_iter):
        skill_new = skill @ T
        skill_new /= skill_new.sum()
        if np.abs(skill_new - skill).max() < tol:
            print(f"  Converged in {it + 1} iterations.")
            return skill_new
        skill = skill_new
    print("  Warning: power method did not fully converge.")
    return skill
 
 
print("\nBuilding Markov transition matrix …")
T = build_markov_transition(nplays, nwins, damping=0.15)
 
print("Running power method …")
skill = power_method(T)

In [ ]:
# Model training

In [ ]:
# Model Evaluation

In [ ]:
# Results etc

# Sources: